# Xenium data perprocessing

In [ ]:
import spatialdata as sd
import spatialdata_plot
from spatialdata_io import xenium

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import squidpy as sq
import pandas as pd

## for memory monitoring
import psutil
import gc,os
from rich import print

# Force garbage collection
gc.collect()  
sc.settings.set_figure_params(dpi=200, facecolor="white")
PATH = "/home/kibr/Work/Vascular_atlas"

## set path and parameters
os.chdir(PATH)
sc.set_figure_params(figsize=(6, 6), frameon=False)

## Loading and save each spatial data to zarr format

In [ ]:
# xenium_path = "./Xenium/20250522__204937__Vascular_SU120523_052225/output-XETG00277__0063394__HIP__20250522__204956"
# xenium_path = "./Xenium/20250522__204937__Vascular_SU120523_052225/output-XETG00277__0063406__IFG__20250522__204956"
# xenium_path = "./Xenium/20250522__204937__Vascular_SU120523_052225/output-XETG00277__0063406__PONS__20250522__204956"

# zarr_path = "./data/Xenium/xenium_HIP.zarr"
# zarr_path = "./data/Xenium/xenium_IFG.zarr"
# zarr_path = "./data/Xenium/xenium_PONS.zarr"

In [ ]:
sdata = xenium(xenium_path)

In [ ]:
## save the data to a zarr format
sdata.write(zarr_path)

### Reload the data will not only make sure the data is correctly saved, but also will increase the speed as the data is stored in zarr format. 

In [ ]:
sdata = sd.read_zarr(zarr_path)
sdata

In [ ]:
## The sdata(zarr) format contain the anndata format tables, which is the standard single cell transcriptome format. 
adata = sdata.tables["table"]
adata

## Calculate the quality control metrics

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)

In [ ]:
cprobes = (
    adata.obs["control_probe_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
cwords = (
    adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
print(f"Negative DNA probe count % : {cprobes}")
print(f"Negative decoding count % : {cwords}")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(
    adata.obs["total_counts"],
    kde=False,
    ax=axs[0],
)

axs[1].set_title("Unique transcripts per cell")
sns.histplot(
    adata.obs["n_genes_by_counts"],
    kde=False,
    ax=axs[1],
)


axs[2].set_title("Area of segmented cells")
sns.histplot(
    adata.obs["cell_area"],
    kde=False,
    ax=axs[2],
)

axs[3].set_title("Nucleus ratio")
sns.histplot(
    adata.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False,
    ax=axs[3],
)

In [ ]:
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)
# adata = adata[adata.obs.n_genes_by_counts < 200, :]
print(adata)

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
## select the resolution
sc.tl.leiden(adata,resolution = 1.5)

## Visualize annotation on UMAP and spatial coordinates

In [ ]:
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(
    adata,
    color=[
        "FLT1",
        "CEMIP",
        "FBLN1",
        # "CARMN",
        "SYT1",
        # "C3",
        # "SLC1A2",
        "HTR2C",
        "MOBP",
        "TNR",
        "total_counts",
        "n_genes_by_counts",
        "leiden",
    ],
    legend_loc="on data",
    wspace=0.4,
)

## Manually cell type annotation based on the cell type marker genes -->

In [ ]:
marker_genes = {
    "Astrocyte": ["SLC1A2", "ADGRV1"],
    "Endothelial": ["FLT1", "MECOM", "ATP10A"],
    "Ependymal": ["HTR2C"],
    "Fibroblast": ["CEMIP", "BICC1", "LAMA2"],
    "Microglia": ["C3", "RYR1"],
    "Mural_Cell": ["CARMN", "MYH11", "RBPMS"],
    "Neuron": ["SYT1", "ROBO2", "CNTNAP2"],
    "Oligodendrocyte": ["MOBP", "ST18", "SLC24A2"],
    "OPC": ["TNR", "DSCAM", "LHFPL3"],
}

sc.pl.dotplot(adata, marker_genes, groupby="leiden", standard_scale="var")

In [ ]:
sc.tl.rank_genes_groups(adata,"leiden",method = "wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False)

In [ ]:
adata.obs["cell_class"] = adata.obs["leiden"].map(
    {
        "0": "Oligodendrocyte",
        "1": "Oligodendrocyte",
        "2": "Oligodendrocyte",
        "3": "Oligodendrocyte",
        "4":"Oligodendrocyte",
        "5":"Unknown",
        "6":"Oligodendrocyte",
        "7":"Astrocyte",
        "8":"Mural_Cell_Fibroblast",
        "9":"Endothelial",
        "10":"Unknown",
        "11":"Microglia",
        "12":"Mural_Cell_Fibroblast",
        "13":"Mural_Cell_Fibroblast",
        "14":"Mural_Cell_Fibroblast",
        "15":"OPC",
    }
)
adata.obs["cell_class"].value_counts()

In [ ]:
ct_palette = {"Oligodendrocyte":"#00BFC4",
              "Microglia":"#DC143C",
              "Neuron":"#08415C",
              "Endothelial":"#fcbe05",
              "Mural_Cell_Fibroblast":"#A26DC2",
              "Astrocyte":"#F06719",
              "OPC":"#0072B2",
              "Unknown":"#5F5E5E",
              }

plt.rcParams["figure.figsize"] = (6, 6)
sc.pl.umap(
    adata,
    color=[
        "cell_class",
    ],
    legend_loc="on data",
    wspace=0.4,
    palette=ct_palette,
)

In [ ]:
## Plot the cell class
plt.rcParams["figure.figsize"] = (8, 8)
adata.uns["cell_class_colors"] = [ct_palette[c] for c in adata.obs["cell_class"].cat.categories]
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None,
    color=[
        "cell_class",
    ],
    wspace=0.4,
)

## Computation of centrality scores

closeness centrality - measure of how close the group is to other nodes.

clustering coefficient - measure of the degree to which nodes cluster together.

degree centrality - fraction of non-group members connected to group members.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)

In [ ]:
sq.gr.centrality_scores(adata, cluster_key="cell_class")

In [ ]:
sq.pl.centrality_scores(adata, cluster_key="cell_class", figsize=(16, 5))

### Compute co-occurrence probability

In [ ]:
sdata.tables["subsample"] = sc.pp.subsample(adata, fraction=0.5, copy=True)
adata_subsample = sdata.tables["subsample"]

In [ ]:
sq.gr.co_occurrence(
    adata,
    cluster_key="cell_class",
)
sq.pl.co_occurrence(
    adata,
    cluster_key="cell_class",
    clusters="Endothelial",
    figsize=(10, 10),
)
# sq.pl.spatial_scatter(
#     adata,
#     color="leiden",
#     shape=None,
#     size=2,
# )

## Neighbors enrichment analysis

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="cell_class")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 8))
sq.pl.nhood_enrichment(
    adata,
    cluster_key="cell_class",
    figsize=(8, 8),
    title="Neighborhood enrichment adata",
    ax=ax[0],
)
sq.pl.spatial_scatter(adata, color="cell_class", shape=None, size=2, ax=ax[1])

## Compute Moran's I score
The Moran’s I global spatial auto-correlation statistics evaluates whether features (i.e. genes) shows a pattern that is clustered, dispersed or random in the tissue are under consideration.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)
sq.gr.spatial_autocorr(
    adata,
    mode="moran",
    n_perms=100,
    n_jobs=1,
)

In [ ]:
adata.uns["moranI"].head(10)

In [ ]:
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    color=[
        "FLT1",
        "SYT1",
    ],
    shape=None,
    size=2,
    img=False,
)

In [ ]:
import spatialdata_plot

gene_name = ["FLT1", "SYT1"]
for name in gene_name:
    sdata.pl.render_images("morphology_focus").pl.render_shapes(
        "cell_circles",
        color=name,
        table_name="table",
        use_raw=False,
    ).pl.show(
        title=f"{name} expression over Morphology image",
        coordinate_systems="global",
        figsize=(10, 5),
    )

In [ ]:
adata.write_h5ad("../data/Xenium/adata/xenium_PONS.h5ad")

### Integration of all data together

In [ ]:
adata1 = sc.read_h5ad("../data/Xenium/adata/xenium_IFG.h5ad")
adata1.obs["Brain_region"] = "IFG"
adata1.obs["Brain_region"].value_counts()
adata1.obs_names = adata1.obs['Brain_region'].astype(str) + "_" + adata1.obs_names.astype(str)

In [ ]:
adata2 = sc.read_h5ad("../data/Xenium/adata/xenium_PONS.h5ad")
adata2.obs["Brain_region"] = "Pons"
adata2.obs["Brain_region"].value_counts()
adata2.obs_names = adata2.obs['Brain_region'].astype(str) + "_" + adata2.obs_names.astype(str)

In [ ]:
adata4 = sc.read_h5ad("../data/Xenium/adata/xenium_HIP.h5ad")
adata4.obs["Brain_region"] = "HIP"
adata4.obs["Brain_region"].value_counts()
adata4.obs_names = adata4.obs['Brain_region'].astype(str) + "_" + adata4.obs_names.astype(str)

In [ ]:
## Concatanation 
adata = sc.concat([adata1,adata2,adata3,adata4],join = "inner")
adata

adata.obs["Brain_region"].value_counts()

### Read the merged object 

In [ ]:
### Reloading
adata = sc.read_h5ad("./Data/Xenium/Xenium/adata/xenium_merged.h5ad")

In [ ]:
adata.obs['Brain_region'].value_counts()

## Excluding CP sample

In [ ]:
subset = adata.obs['Brain_region'].isin(["HIP","IFG","Pons"])
adata = adata[subset].copy()

In [ ]:
print(adata.layers["counts"][1:6,1:6])

## Final round of cell annotation for confirming

In [ ]:
## Preprocessing again for checking
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata, inplace=True,target_sum = 10**4)
sc.pp.log1p(adata)

# sc.pp.regress_out(adata, ["total_counts"])
# sc.pp.scale(adata)

sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
marker_genes = {
    "Astrocyte": ["SLC1A2", "ADGRV1"],
    "Endothelial": ["FLT1", "MECOM", "ATP10A","SLC7A5"],
    "Ependymal": ["HTR2C"],
    "Fibroblast": ["CEMIP", "BICC1", "LAMA2"],
    "Microglia": ["C3", "RYR1"],
    "Mural_Cell": ["CARMN", "MYH11", "RBPMS"],
    "Neuron": ["SYT1", "ROBO2", "CNTNAP2"],
    "Oligodendrocyte": ["MOBP", "ST18", "SLC24A2"],
    "OPC": ["TNR", "DSCAM", "LHFPL3"],
}

sc.pl.dotplot(adata, marker_genes, groupby="cell_class", standard_scale="var")

In [ ]:
# Replace Pericyte and SMC with Mural cell
# adata.obs['cell_type'] = adata.obs['cell_type'].replace({
#     'Meningeal_Fibroblast': 'Fibroblast',
#     'Perivascular_Fibroblast': 'Fibroblast'
# })
adata.obs['cell_type'].value_counts()

In [ ]:
ct_palette = {"Oligodendrocyte":"#00BFC4",
              "Microglia":"#DC143C",
              "Neuron":"#08415C",
              "Endothelial":"#fcbe05",
              "Mural_Cell_Fibroblast":"#A26DC2",
              "Astrocyte":"#F06719",
              "OPC":"#0072B2",
              "Unknown":"#5F5E5E",
              }


In [ ]:
sc.set_figure_params(vector_friendly=True, dpi_save=400)
# plt.rcParams["figure.figsize"] = (6, 6)
sc.set_figure_params(figsize=(8, 8), frameon=False)

sc.pl.umap(
    adata,
    # color=[
    #     "cell_type",'leiden','Brain_region'
    # ],
    color=['cell_class'],
    legend_loc="on data",
    wspace=0.4,
    return_fig = False,
    size = 5,
    show=True,
    palette=ct_palette,
    save=f"_Xenium_CC_UMAP.svg",

)

sc.pl.umap(
    adata,
    # color=[
    #     "cell_type",'leiden','Brain_region'
    # ],
    color=['cell_type','Brain_region'],
    legend_loc="on data",
    wspace=0.4,
    return_fig = False,
    size = 5,
    show=True,
    # palette=ct_palette,
    save=f"_Xenium_CT_UMAP.svg",
)

# Save as rasterized PDF
# plt.savefig("../Results/Figures/NicheCompass/umap_cell_type_annotation.pdf", dpi=300, format="pdf", bbox_inches="tight")
# plt.show()    

In [ ]:
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(
    adata,
    color=[
        "FLT1",
        "CEMIP",
        "FBLN1",
        "CARMN",
        "SYT1",
        "C3",
        "SLC1A2",
        "HTR2C",
        "MOBP",
        "TNR",
        "Brain_region",
        "cell_type",
        # "leiden",
    ],
    legend_loc="on data",
    wspace=0.4,
)


## cell type annotation

In [ ]:
### Manually annotation
## select the resolution
sc.tl.leiden(adata,resolution = 2)

In [ ]:
sc.pl.umap(
    adata,
    color=[
        "Brain_region",
    ],
    wspace=0.4,
)

sc.pl.umap(
    adata,
    color=[
        "leiden",
    ],
    legend_loc="on data",
    wspace=0.4,
)

In [ ]:
sc.tl.rank_genes_groups(adata,"leiden",method = "wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False)

In [ ]:
results_df = sc.get.rank_genes_groups_df(adata, group='15',)
results_df.head(15)

In [ ]:
adata.obs["cell_type"] = adata.obs["leiden"].map(
    {
        "0":"Neuron_1",
        "1":"Oligodendrocytes",
        "2":"Mixed",
        "3":"Endothelial",
        "4":"Oligodendrocytes",
        "5":"Neuron_1",
        "6":"Oligodendrocytes",
        "7":"Microglia",
        "8":"Oligodendrocytes",
        "9":"Astrocyte",
        "10":"Neuron_2",
        "11":"Mixed",
        "12":"Mixed",
        "13":"SMC",
        "14":"Pericyte", 
        "15":"Meningeal_Fibroblast",
        "16":"OPC",
        "17":"Neuron_2",
        "18":"Oligodendrocytes",
        "19":"Neuron_2",
        "20":"Neuron_3",
        "21":"Perivascular_Fibroblast",
        "22":"Microglia",
        "23":"Astrocyte",
        "24":"Mixed",
        "25":"Endothelial",
        "26":"Microglia",
        "27":"OPC",
    }
)
adata.obs["cell_type"].value_counts()

In [ ]:
plt.rcParams["figure.figsize"] = (6, 6)
sc.pl.umap(
    adata,
    color=[
        "cell_type",
    ],
    wspace=0.4,
)

In [ ]:
marker_genes = {
    "Astrocyte": ["SLC1A2", "ADGRV1"],
    "Endothelial": ["FLT1", "VWF", "CLDN5"],
    "Fibroblast": ["FBLN1","CEMIP"],
    "Microglia": ["C3", "C1QA"],
    "Neuron": ["SYT1", "ATP1A3","GRIA1","PVALB"],
    "OPC": ["TNR", "DSCAM", "LHFPL3"],
    "Oligodendrocyte": ["MOBP", "SLC44A1"],
    "Mural_Cell": ["CARMN", "MYH11", "RBPMS"],
}

sc.pl.dotplot(adata, marker_genes, groupby="cell_type", standard_scale="var",show = False)
# Save as rasterized PDF
plt.savefig("./Results/Revision_2/Xenium/Xenium_dot_plot.pdf", dpi=300, format="pdf", bbox_inches="tight")
plt.show()     

In [ ]:
adata.write_h5ad("./Data/Xenium/Xenium/adata/xenium_merged.h5ad")
print(adata)

In [ ]:
adata.obs['cell_type'].value_counts()

In [ ]:
ct_palette = {
    "Oligodendrocytes": "#1f78b4",   # vivid blue
    "Neuron_1": "#ffb000",          # vivid amber
    "Mixed": "#7f7f7f",             # gray (unchanged)
    "Endothelial": "#e60026",       # bright red (highlight)
    "Microglia": "#33a02c",         # vibrant green
    "Astrocyte": "#6a3d9a",         # deep purple
    "Neuron_2": "#ff7f0e",          # strong orange
    "Neuron_3": "#1aa1a1",          # cyan-teal
    "OPC": "#b15928",               # earthy brown
    "SMC": "#fb9a99",               # light rose pink
    "Pericyte": "#a6cee3",          # pale blue
    "Fibroblast": "#cab2d6",  # soft lavender
}
adata.uns["cell_type_colors"] = [ct_palette[c] for c in adata.obs["cell_type"].cat.categories]

colors = adata.obs['cell_type'].map(ct_palette)

In [ ]:
spatial_coord = pd.DataFrame(adata.obsm["spatial"],
                             index = adata.obs_names,
                             columns = ["spatial_x","spatial_y"])

adata.obs[["spatial_x","spatial_y"]] = spatial_coord

In [ ]:
# adata.obs.query("Brain_region == 'CP'").plot.scatter(
#     x="spatial_x", 
#     y="spatial_y",
#     c="cell_type", 
#     cmap="tab20",
#     s = 1
# )

# adata.obs.query("Brain_region == 'HIP'").plot.scatter(
#     x="spatial_x", 
#     y="spatial_y",
#     c="cell_type", 
#     cmap="tab20",
#     s = 1
# )

# adata.obs.query("Brain_region == 'IFG'").plot.scatter(
#     x="spatial_x", 
#     y="spatial_y",
#     c="cell_type", 
#     cmap="tab20",
#     s = 1
# )

# adata.obs.query("Brain_region == 'Pons'").plot.scatter(
#     x="spatial_x", 
#     y="spatial_y",
#     c="cell_type", 
#     cmap="tab20",
#     s = 1
# )

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)

# subset = adata.obs['Brain_region'] == "CP"
# adata_subset = adata[subset].copy()
# sq.pl.spatial_scatter(adata_subset, color="cell_type", shape=None, size=2)

subset = adata.obs['Brain_region'] == "HIP"
adata_subset = adata[subset].copy()
sq.pl.spatial_scatter(adata_subset, color="cell_type", shape=None, size=8,save=f"_Xenium_HIP_spatial_scatter.svg")

subset = adata.obs['Brain_region'] == "Pons"
adata_subset = adata[subset].copy()
sq.pl.spatial_scatter(adata_subset, color="cell_type", shape=None, size=8,save=f"_Xenium_Pons_spatial_scatter.svg")

subset = adata.obs['Brain_region'] == "IFG"
adata_subset = adata[subset].copy()
sq.pl.spatial_scatter(adata_subset, color="cell_type", shape=None, size=8,save=f"_Xenium_IFG_spatial_scatter.svg")


### Subsetting the object and explore the neighborhood of each nuclei.

In [ ]:
subset = adata.obs['Brain_region'] == "HIP"
adata_subset = adata[subset].copy()
# sq.pl.spatial_scatter(adata_subset, color="cell_type", shape=None, size=2)

In [ ]:
sq.gr.spatial_neighbors(adata_subset, coord_type="generic", delaunay=True)
sq.gr.nhood_enrichment(adata_subset, cluster_key="cell_type")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 8))
sq.pl.nhood_enrichment(
    adata_subset,
    cluster_key="cell_type",
    figsize=(8, 8),
    title="Neighborhood enrichment HIP",
    ax=ax[0],
)
sq.pl.spatial_scatter(adata_subset, color="cell_type", shape=None, size=2, ax=ax[1])
## save plot as rasterized PDF
plt.savefig("./Results/Revision_2/Xenium/HIP_Nhood_enrichment.pdf", dpi=300, format="pdf", bbox_inches="tight")
plt.show()    

# Saving the merged object.

In [ ]:
adata.write_h5ad("./Data/Xenium/Xenium/adata/xenium_merged.h5ad")

### Subsetting the cluster belong to vascular

In [ ]:
plt.rcParams["figure.figsize"] = (6,6)
sc.pl.umap(
    adata,
    color=[
        "leiden",
        "cell_type"
    ],
    legend_loc="on data",
    wspace=0.4,
)

In [ ]:
## Subsetting the object by keeping the vascular nuclei only
subset = adata.obs["leiden"].isin(["6","19","27"])
adata_vascular = adata[subset].copy()
adata_vascular

In [ ]:
subset = adata_vascular.obs["leiden"].isin(["7","8"])
adata_vascular = adata_vascular[~subset].copy()
adata_vascular

### Redo the clustering and subtyping

In [ ]:
adata_vascular = sc.read_h5ad("./Results/Endo_Xenium.h5ad")
adata_vascular

In [ ]:
print(adata_vascular.layers["counts"][:7,:7])

In [ ]:
## Preprocessing again for checking
adata_vascular.X = adata_vascular.layers["counts"].copy()
sc.pp.normalize_total(adata_vascular, inplace=True)
sc.pp.log1p(adata_vascular)

# sc.pp.regress_out(adata_vascular, ["transcript_counts"])
# sc.pp.scale(adata_vascular)

sc.pp.pca(adata_vascular)
sc.pp.neighbors(adata_vascular)
sc.tl.umap(adata_vascular)

sc.tl.leiden(adata_vascular,resolution = 1)

In [ ]:
sc.settings.figdir = f"{PATH}/Results/Revision_2/Xenium/"
plt.rcParams["figure.figsize"] = (6,6)
sc.pl.umap(
    adata_vascular,
    color=[
        "leiden",
        "Brain_region"
    ],
    legend_loc="on data",
    wspace=0.4,
    size = 12,
    save=f"_Xenium_Endo_UMAP_clean.svg",
)

In [ ]:
sc.tl.rank_genes_groups(adata_vascular,"leiden",method = "wilcoxon")
sc.pl.rank_genes_groups(adata_vascular, n_genes=20, sharey=False)

In [ ]:
# ## remove a few clusters that might be doublets or low quality
# subset = adata_vascular.obs["leiden"].isin(["6"])
# adata_vascular = adata_vascular[~subset].copy()

In [ ]:
marker_genes = {
    "Large Artery": ["DKK2","PELI1"],
    "Arterial": ["NOS1","ELN"],
    "Arteriole": ["ARL15","VEGFC"],
    "Capillary1": ["SLC7A5","SLC39A10","ST6GAL1","SLC16A1","FLT1", "VWF", "CLDN5"],
    "Capillary2": ["PCDH9", "GPM6A"],
    "Capillary3": ["SLC26A3","DLC1"],
    "Venule": ["TSHZ2","IL1R1"],
    "Vein": ["PVALB","GRIA1"],
    "EndoMT1": ["CACNA1C"],
    "Microglia": ["C3"],
    "Neuron":["SYT1","ROBO2"],
    "Oligo":["MOBP"]
}

sc.pl.dotplot(adata_vascular, marker_genes, groupby="leiden", standard_scale="var")

In [ ]:
adata_vascular.obs["cell.type"] = adata_vascular.obs["leiden"].map(
    {
        "0":"Capillary",
        "1":"Arterial",
        "2":"Venous",
        "3":"Arterial",
        "4":"Capillary",
        "5":"Venous",
        "6":"Arterial",
        "7":"Capillary",
        "8":"Capillary",
    }
)
adata_vascular.obs["cell.type"].value_counts()


adata_vascular.obs["cell.type"].value_counts()

In [ ]:
marker_genes = {
    "Arterial": ["DKK2","ARL15","VEGFC"],
    "Capillary": ["SLC7A5","SLC16A1","FLT1","CLDN5"],
    "Venous": ["VWF","TSHZ2","IL1R1"],
    "Others": ["CACNA1C"],
}
sc.pl.dotplot(adata_vascular, marker_genes, groupby="cell.type", standard_scale="var",show = False,save=f"_Xenium_Endodot_plot.svg")

# Save as rasterized PDF
# plt.savefig("../Results/Figures/NicheCompass/Xenium_vascular_dot_plot.pdf", dpi=300, format="pdf", bbox_inches="tight")
plt.show()    

In [ ]:
sc.pl.umap(
    adata_vascular,
    color=[
        "Brain_region","cell.type"
    ],
    wspace=0.4,
    return_fig = bool,
    size = 15,
    show=False,
    save=f"_Xenium_Endo_UMAP_clean.svg",
)
# Save as rasterized PDF
# plt.savefig("../Results/Figures/NicheCompass/umap_endo.pdf", dpi=300, format="pdf", bbox_inches="tight")
plt.show()    

### Integrate with existing snRNA-seq data and see if fits in existing dataset

In [ ]:
## import modules
import os
import scvi
import numpy as np
import anndata as ad
import torch

In [ ]:
## setting randomization seed
scvi.settings.seed = 42
print("Last run with scvi-tools version:", scvi.__version__)

In [ ]:
# Check if CUDA is available
if torch.cuda.is_available():
    print("CUDA is available")
    print(torch.version.cuda)
    # Get the number of available GPUs
    num_gpus = torch.cuda.device_count()
    print(f"Number of available GPUs: {num_gpus}")
    
    # Get the name of each available GPU
    for i in range(num_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        print(f"GPU {i}: {gpu_name}")
else:
    print("CUDA is not available")

In [ ]:
## general setting
sc.settings.set_figure_params(dpi=50, facecolor="white")
# os.chdir(path)
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()
torch.set_float32_matmul_precision("high")

In [ ]:
## Load the AnnData
## The data was annotated using celltypist annotation  
adata_endo = sc.read("./Results/Revision_2/Endo_temp_processed.h5ad") # merged dataset
# print(adata_endo)
adata_endo.obs["modality"] = "snRNAseq"
adata_endo.obs["cell.type"] = adata_endo.obs["Cell_type"].copy()
print(adata_endo)

In [ ]:
sc.pl.embedding(adata_endo, basis='umap_harmony', color=['cell.type',"brain_region"])

In [ ]:
adata_vascular.X = adata_vascular.layers["counts"].copy()

adata_vascular.obs["brain_region"] = adata_vascular.obs["Brain_region"].copy()
adata_vascular.obs["modality"] = "Xenium"
adata_vascular.obs["individualID"] = "Stanford_12052023"

sc.pl.umap(adata_vascular, color=['cell.type',"brain_region"])

### Get the same genes and used for reference data

In [ ]:
genes = adata_vascular.var.index

In [ ]:
adata_endo = adata_endo[:,genes].copy()
adata_endo

In [ ]:
adata_endo.obs["labels_scanvi"] = adata_endo.obs["cell.type"].values

### Backup the raw read counts
adata_endo.layers["counts"] = adata_endo.X.copy()

In [ ]:
## normalization
adata_endo.X = adata_endo.layers["counts"].copy()

sc.pp.normalize_total(adata_endo)
sc.pp.log1p(adata_endo)

## dimensionality reduction
sc.tl.pca(adata_endo)
sc.pp.neighbors(adata_endo) 
sc.tl.umap(adata_endo)

In [ ]:
# sc.set_figure_params(figsize=(5, 5), frameon=False)
sc.pl.umap(
    adata_endo,
    color=["individualID","modality","brain_region","cell.type"],
    # Setting a smaller point size to get prevent overlap
    ncols=2,
    size=2,
    # save="_00_scanpy_unintegrated_all_0121.svg"
)

In [ ]:
adata_endo.layers['log1p'] = adata_endo.X.copy()

In [ ]:
## uing log1p for integration tasks
scvi.model.SCVI.setup_anndata(
    adata_endo,
    layer="log1p",
    batch_key="individualID",
    # labels_key = "cell.type",
    # categorical_covariate_keys=['individualID'],
    # continuous_covariate_keys=['individualID']
    )

scvi_model = scvi.model.SCVI(adata_endo, n_layers=2, n_latent=30, gene_likelihood="nb")
# scvi_model.view_anndata_setup()

In [ ]:
## check how many epochs needed for 
import numpy as np
max_epochs_scvi = np.min([round((20000 / adata_endo.n_obs) * 400), 400])
max_epochs_scvi

In [ ]:
## train scvi model first for integration
scvi_model.train() #batch_size=125
# scvi_model.save(dir_path = PATH+"/Results/Revision_2/scvi_model_cellranger/",overwrite = True) ## save model

In [ ]:
## assign the scvi latent key to the anndata 
SCVI_LATENT_KEY = "X_scVI"
adata_endo.obsm[SCVI_LATENT_KEY] = scvi_model.get_latent_representation()

## compute the umap and do the plotting
# SCVI_LATENT_KEY = "X_scVI_sampleID"
sc.pp.neighbors(adata_endo, use_rep=SCVI_LATENT_KEY)
sc.tl.leiden(adata_endo,resolution=1,key_added='leiden1')
sc.tl.umap(adata_endo)

In [ ]:
## plotting
sc.pl.umap(
    adata_endo,
    color=["individualID","cell.type","brain_region","modality"],
    frameon=False,
    ncols=2,
    size=10,
    legend_loc="on data",
    # save = f"_01_SCVI_cellranger.svg"
)

### SCANVI model for cell label transferring

In [ ]:
adata_endo.obs["labels_scanvi"] = adata_endo.obs["cell.type"].values

In [ ]:
scanvi_model = scvi.model.SCANVI.from_scvi_model(
    scvi_model,
    unlabeled_category="Unknown",
    labels_key="labels_scanvi",
)

In [ ]:
## As the scANVI is more like a fine tuning, less epochs needed.
max_epochs_scanvi = int(np.min([10, np.max([2, round(max_epochs_scvi / 3.0)])]))
print(max_epochs_scanvi)
## using the parameter from the tutorial
scanvi_model.train(max_epochs=20, early_stopping=True)
scanvi_model.save(dir_path = PATH+"/Results/Revision_2/scanvi_model_cellranger/",overwrite = True)

In [ ]:
## give label
SCANVI_LATENT_KEY = "X_scANVI"
adata_endo.obsm[SCANVI_LATENT_KEY] = scanvi_model.get_latent_representation(adata_endo)
sc.pp.neighbors(adata_endo, use_rep=SCANVI_LATENT_KEY)
sc.tl.leiden(adata_endo,resolution=1,key_added='leiden1')
sc.tl.umap(adata_endo)

In [ ]:
## check the umap plotting integration again for different leiden space
sc.set_figure_params(figsize=(7, 7), frameon=False)

## plotting
sc.pl.umap(
    adata_endo,
    color=["individualID","cell.type","brain_region","modality"],
    frameon=False,
    ncols=2,
    size=4,
    legend_loc="on data",
    save = "_02_scANVI_integration_cellranger.svg"
)

### Query data

In [ ]:
### Backup the raw read counts
adata_vascular.layers["counts"] = adata_vascular.X.copy()

In [ ]:
print(adata_vascular.X[:7,:8])
print(adata_vascular.layers["counts"][:7,:8])
# print(adata_vascular)

In [ ]:
## normalization
adata_vascular.X = adata_vascular.layers["counts"].copy()

sc.pp.normalize_total(adata_vascular)
sc.pp.log1p(adata_vascular)

## dimensionality reduction
sc.tl.pca(adata_vascular)
sc.pp.neighbors(adata_vascular)
sc.tl.umap(adata_vascular)

In [ ]:
adata_vascular.layers['log1p'] = adata_vascular.X.copy()

In [ ]:
sc.pl.umap(adata_vascular, color=['cell.type',"brain_region"])

## preparing the model

In [ ]:
model_path = PATH+"/Results/Revision_2/scanvi_model_cellranger/"

scvi.model.SCANVI.prepare_query_anndata(adata_vascular, model_path)

In [ ]:
query_model = scvi.model.SCANVI.load_query_data(
    adata_vascular,
    model_path,
)

In [ ]:
## train query model
query_model.train(
    max_epochs=100,
    plan_kwargs=dict(weight_decay=0.0),
    check_val_every_n_epoch=10,
)

In [ ]:
SCANVI_LATENT_KEY = "X_scANVI"

adata_vascular.obsm[SCANVI_LATENT_KEY] = query_model.get_latent_representation()
adata_vascular.obs["predictions1"] = query_model.predict()
# adata_vascular.obs["cell.type"] = adata_vascular.obs["predictions"].copy()

sc.pp.neighbors(adata_vascular, use_rep=SCANVI_LATENT_KEY)
sc.tl.leiden(adata_vascular)
sc.tl.umap(adata_vascular)

In [ ]:
## check the umap plotting integration again for different leiden space
sc.set_figure_params(figsize=(7, 7), frameon=False)

## plotting
sc.pl.umap(
    adata_vascular,
    color=["cell.type","predictions1","brain_region","modality"],
    frameon=False,
    ncols=2,
    size=10,
    # legend_loc="on data",
    save = f"_03_scANVI_query.svg"
)

### Analyze together

In [ ]:
adata_concat = ad.concat([adata_endo,adata_vascular])
adata_concat

In [ ]:
adata_concat = adata_vascular.concatenate(adata_endo)
# adata_concat.obs["predictions1"].value_counts()

In [ ]:
full_predictions = query_model.predict(adata_concat)
print(f"Acc: {np.mean(full_predictions == adata_concat.obs["cell.type"])}")

adata_concat.obs["predictions"] = full_predictions

In [ ]:
sc.pp.neighbors(adata_concat, use_rep=SCANVI_LATENT_KEY)
sc.tl.leiden(adata_concat)
sc.tl.umap(adata_concat)

In [ ]:
## check the umap plotting integration again for different leiden space
sc.set_figure_params(figsize=(8, 8), frameon=False)

## plotting
sc.pl.umap(
    adata_concat,
    color=["individualID","cell.type","brain_region","modality","predictions"],
    frameon=False,
    ncols=2,
    size=4,
    legend_loc="on data",
    save = f"_04_scANVI_integration.svg"
)

## Run SCVI for integration again?

In [ ]:
## uing log1p for integration tasks
scvi.model.SCVI.setup_anndata(
    adata_concat,
    layer="log1p",
    batch_key="modality",
    categorical_covariate_keys=['individualID'],
    # continuous_covariate_keys=['total_counts', 'n_genes_by_counts','pct_counts_mt','pct_counts_ribo']
    )

scvi_model = scvi.model.SCVI(adata_concat, n_layers=2, n_latent=30, gene_likelihood="nb")

In [ ]:
## check how many epochs needed for 
import numpy as np
max_epochs_scvi = np.min([round((20000 / adata.n_obs) * 400), 400])
max_epochs_scvi

In [ ]:
## train scvi model first for integration
scvi_model.train() #batch_size=125
scvi_model.save(dir_path = PATH+"/Results/Revision_2/scvi_model_cellranger/",overwrite= True) ## save model

In [ ]:
## assign the scvi latent key to the anndata 
SCVI_LATENT_KEY = "X_scVI"
adata_concat.obsm[SCVI_LATENT_KEY] = scvi_model.get_latent_representation()

In [ ]:
## compute the umap and do the plotting
# SCVI_LATENT_KEY = "X_scVI_sampleID"
sc.pp.neighbors(adata_concat, use_rep=SCVI_LATENT_KEY)
sc.tl.leiden(adata_concat,resolution=1,key_added='leiden1')
sc.tl.umap(adata_concat)

In [ ]:
## plotting
sc.pl.umap(
    adata_concat,
    color=["individualID","modality","cell.type","brain_region","predictions"],
    frameon=False,
    size = 5,
    ncols=2,
    legend_loc="on data",
    save = f"_05_SCVI_integration.svg"
)

In [ ]:
ax = sc.pl.umap(
    adata_concat,
    frameon=False,
    show=False,
    size = 30
)
sc.pl.umap(
    adata_concat[: adata_vascular.n_obs],
    color=["cell.type"],
    frameon=False,
    title="Xenium cell.type",
    ax=ax,
    alpha=0.7,
    size = 20,
    save = f"_06_Xenium_celltype_UMAP.svg"
)

ax = sc.pl.umap(
    adata_concat,
    frameon=False,
    show=False,
    size = 30
)
sc.pl.umap(
    adata_concat[: adata_endo.n_obs],
    color=["cell.type"],
    frameon=False,
    title="snRNAseq cell.types",
    ax=ax,
    alpha=0.7,
    size = 20,
    save = f"_07_snRNAseq_celltype_UMAP.svg"
)

## Saving data

In [ ]:
adata_concat.write_h5ad("./Results/all_endo.h5ad")

In [ ]:
subset = adata_concat.obs["modality"] == "Xenium"
adata_subset = adata_concat[subset].copy()
adata_subset.obs["modality"].value_counts()

In [ ]:
adata_subset.write_h5ad("./Results/Endo_Xenium_clean.h5ad")

In [ ]:
adata_subset

## Reloading the whole dataset and subsetted dataset for final clean

In [ ]:
adata = sc.read_h5ad(f"{PATH}/Data/Xenium/adata/xenium_merged.h5ad")
print(adata)

adata_endo = sc.read_h5ad("./Results/Endo_Xenium_clean.h5ad")
# print(adata_endo)

In [ ]:
print(adata.obs["cell_class"].value_counts())
print(adata.obs["cell_type"].value_counts())
print(adata_endo.obs["cell_type"].value_counts())
print(adata_endo.obs["cell.type"].value_counts())

In [ ]:
adata_endo.obs_names = adata_endo.obs_names.str.replace("-0", "", regex=False)
adata_endo.obs_names

common_cells = adata.obs_names.intersection(adata_endo.obs_names)
common_cells

In [ ]:
# Get union of categories
all_cats = adata.obs["cell_type"].cat.categories.union(
    adata_endo.obs["cell.type"].cat.categories
)

# Set the unified categories in adata
adata.obs["cell_type"] = adata.obs["cell_type"].cat.set_categories(all_cats)
adata_endo.obs["cell.type"] = adata_endo.obs["cell.type"].cat.set_categories(all_cats)

# Now assign safely
adata.obs.loc[common_cells, "cell_type"] = adata_endo.obs.loc[common_cells, "cell.type"]
adata.obs["cell_type"].value_counts()

In [ ]:
adata.obs["cell_type"] = adata.obs["cell_type"].astype(str)
adata.obs.loc[adata.obs["cell_type"].isin(["Neuron_1","Neuron_2","Neuron_3"]), "cell_type"] = "Neuron"
adata.obs.loc[adata.obs["cell_type"].isin(["Meningeal_Fibroblast","Perivascular_Fibroblast"]), "cell_type"] = "Fibroblast"
adata.obs.loc[adata.obs["cell_type"].isin(["Endothelial"]), "cell_type"] = "Mixed"
adata.obs["cell_type"].value_counts()

In [ ]:
## Check the adata object and clean up the obs names
print(adata)
obs_removs = ['predicted_labels', 'majority_voting', 'conf_score', 'celltypist_cell_label_MTG', 'celltypist_conf_score_MTG', 
'celltypist_cell_label_Vascular', 'celltypist_conf_score_Vascular', 'celltypist_cell_label_PFC', 
'celltypist_conf_score_PFC', 'over_clustering', 'celltypist_cell_label_AY', 'celltypist_conf_score_AY',"cell_class"]

for obs in obs_removs:
    if obs in adata.obs.columns:
        adata.obs.drop(columns=obs, inplace=True)   
print(adata)

In [ ]:
cell_type_colors = {
    "Oligodendrocytes": "#00BFC4",  
    "Mixed": "#7f7f7f",             
    "Neuron": "#08415C",            
    "Microglia": "#d62728",         
    "Astrocyte": "#F06719",         
    "OPC": "#0072B2",                 
    "Capillary": "#66C2A5",        
    "Venous": "#4393C3",           
    "Pericyte": "#e377c2",          
    "SMC": "#9467bd",               
    "Arterial": "#D6604D",         
    "Fibroblast": "#5b844d"         
}

ordered_cell_types = [
    "Astrocyte", 
    "Arterial",
    "Capillary",
    "Venous",
    "Fibroblast",
    "Microglia",
    "Neuron",
    "OPC",
    "Oligodendrocytes",
    "Pericyte",
    "SMC",
    "Mixed",
]

In [ ]:
## clean umap showing the cell type and brain region
sc.settings.figdir = f"{PATH}/Results/Revision_2/Xenium/"
sc.set_figure_params(figsize=(7, 7), frameon=False)

In [ ]:
sc.pl.embedding(
    adata,
    basis="X_umap",
    color=["cell_type"],
    palette=cell_type_colors,
    frameon=False,
    ncols=2,
    size=4,
    legend_loc="on data",
    save = f"umap_Xenium_CT_UMAP.svg"
)

sc.pl.embedding(
    adata,
    basis="X_umap",
    color=["Brain_region"],
    # palette=cell_type_colors,
    frameon=False,
    ncols=2,
    size=4,
    legend_loc="on data",
    save = f"umap_Xenium_Brain_region_UMAP.svg"
)

In [ ]:
## Final dotplot for the marker genes
marker_genes = {
    "Astrocyte": ["SLC1A2", "ADGRV1"],
    "Endothelial": ["ARL15","SLC7A5", "VWF","TSHZ2","FLT1","CLDN5"],
    "Fibroblast": ["FBLN1","CEMIP"],
    "Microglia": ["C3", "C1QA"],
    "Neuron": ["SYT1", "ATP1A3","GRIA1","PVALB"],
    "OPC": ["TNR", "DSCAM", "LHFPL3"],
    "Oligodendrocyte": ["MOBP", "SLC44A1"],
    "Mural_Cell": ["CARMN", "MYH11", "RBPMS"],
}
## Change the plot cell type order
adata.obs["cell_type"] = pd.Categorical(adata.obs["cell_type"], categories=ordered_cell_types, ordered=True)
sc.pl.dotplot(
    adata, marker_genes, 
    groupby="cell_type", 
    standard_scale="var",
    save="cell_type_marker_genes_dotplot.svg"
    )

In [ ]:
## save the final anndata object for downstream analysis
adata.write_h5ad("./Data/Xenium/adata/Xenium_final_adata.h5ad")